# Visualizing Alignment Scores

The fundamental visualization objects are
* an array of `VerseScore` instances, loaded into a Pandas DataFrame
* Altair heatmap, like [Simple Heatmap — Vega-Altair 5.5.0 documentation](https://altair-viz.github.io/gallery/simple_heatmap.html)

`VerseScore.asdict()` produces a dictionary like this:

```
{'Identifier': '40001001',
 'Verse': '001',
 'Chapter': '40001',
 'Book': '40',
 'Reference': 'MAT 1:1',
 'AER': 0.0,
 'F1': 0.615,
 'Precision': 1.0,
 'Recall': 0.444}
```

Any of these attributes can be used in visualization, as well as filtering a `GroupScore` instance. 

## Score the Data

Fuller details in [Scoring Alignment Data](31ScoringAlignmentData.ipynb].

In [1]:
import altair as alt
import pandas as pd
# don't omit columns for wide dataframes
pd.set_option('display.max_columns', None)

# import with an abbreviated alias
import biblealignlib as bal
from biblealignlib.burrito import AlignmentSet, util
from biblealignlib.autoalign import Scorer

targetlang, targetid, sourceid = ("eng", "BSB", "SBLGNT")
alsetref = AlignmentSet(targetlanguage=targetlang,
        targetid=targetid,
        sourceid=sourceid,
        langdatapath=(bal.CLEARROOT / f"alignments-{targetlang}/data"))

In [2]:
# this directory should already exist and have burrito format alignments
condition = "notebookcheck"
conditiondir = alsetref.langdatapath.parent / f"exp/{targetid}/{condition}"
print(f"Conditiondir: {conditiondir}")
sc = Scorer(referenceset=alsetref, 
                   hypothesispath=(conditiondir / f"{sourceid}-{targetid}-eflomal.json"),
                   hypothesisaltid="eflomal")

Conditiondir: /Users/sboisen/git/Clear-Bible/alignments-eng/exp/BSB/notebookcheck

        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/targets/BSB/nt_BSB.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/SBLGNT-BSB-manual.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/SBLGNT-BSB-manual.toml
        
----- Hypothesis data: <AlignmentSet: eng, SBLGNT-BSB-eflomal>

        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/targets/BSB/nt_BSB.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-eng/exp/BSB/notebookcheck/SBLGNT-BSB-eflomal.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/SBLGNT-BSB-manual.toml
        
Dropping 14346 bad

## Collect Chapter x Verse Scores for a Book



In [11]:
# the maximum items Altair can handle is 5000, so can't do the whole NT this way
mrkscore = sc.score_group(identifier="41")
print(mrkscore.summary())
mrkscore

4: AER=0.1773	P=0.8227	R=0.5316	F1=0.6459


<GroupScore: 4>

In [32]:
matactscore = sc.score_partial(startbcv="40001001", endbcv="44028031")

## Visualize as a Heatmap

In [33]:
source = pd.DataFrame([vs.asdict() for vs in matactscore.verse_scores])
#source = pd.DataFrame([vs.asdict() for vs in mrkscore.verse_scores])
source.head()

,Identifier,Verse,Chapter,Book,Reference,AER,F1,Precision,Recall
0,40001001,001,40001,40,MAT 1:1,0.000,0.615,1.000,0.444
1,40001002,002,40001,40,MAT 1:2,0.154,0.667,0.846,0.550
2,40001003,003,40001,40,MAT 1:3,0.474,0.500,0.526,0.476
3,40001004,004,40001,40,MAT 1:4,0.000,0.692,1.000,0.529
4,40001005,005,40001,40,MAT 1:5,0.706,0.270,0.294,0.250


In [34]:
alt.Chart(source, title="MRK-ACT F1 Scores").mark_rect().encode(
    x='Verse:O',
    y='Chapter:O',
    color='F1:Q',
    tooltip=['Reference', 'F1', 'Precision', 'Recall']
)

alt.Chart(...)

In [35]:
# subset of verses with low F1 scores
lowf1 = source[source.F1 < 0.2]
alt.Chart(lowf1, title="Precision for Low F1 Scores").mark_rect().encode(
    x='Verse:O',
    y='Chapter:O',
    color='Precision:Q',
    tooltip=['Reference', 'F1', 'Precision', 'Recall']
)

alt.Chart(...)

## Verses with Low Recall Scores

If precision is high and recall is low, the aligner is failing to generate enough hypotheses. How come?

In [36]:
# subset of verses with low recall scores
lowrecall = source[source.Recall < 0.2]
alt.Chart(lowrecall, title="Precision for Low Recall Scores").mark_rect().encode(
    x='Verse:O',
    y='Chapter:O',
    color='Precision:Q',
    tooltip=['Reference', 'Identifier', 'F1', 'Precision', 'Recall']
)

alt.Chart(...)

A few notable examples with high precision despite low recall:

* LUK 7:18: BSB puts the equivalent of "And summoning a certain two of his disciples, John ..." in the next verse.
* JHN 18:30: some large alignment records here, both in reference and hypothesis


In [37]:
sc.verse_dataframe("42007018")

,Then,John’s,disciples,informed,him,about,all,these,things
Καὶ,R-H,,,,,,,,
ἀπήγγειλαν,,,,R-H,,,,,
Ἰωάννῃ,,,,,R--,,,,
οἱ,,,,,,,,,
μαθηταὶ,,,R-H,,,,,,
αὐτοῦ,,R--,,,--H,,,,
περὶ,,,,,,R-H,,,
πάντων,,,,,,,R-H,,
τούτων,,,,,,,,R-H,R--
καὶ,,,,,,,,R--,R--


In [38]:
sc.verse_dataframe("43018030")

,If,He,were,not,a,criminal,they,replied,we,would,not.1,have,handed,Him,over,to,you
ἀπεκρίθησαν,,,,,,,R--,R--,,,,,,,,,
καὶ,,,,,,,R-H,R--,,,,,,,,,
εἶπαν,,,,,,,R--,R--,,,,,,,,,
αὐτῷ,,,,,,,R--,R--,,,,,,,,,
Εἰ,R-H,,,,,,,,,,,,,,,,
μὴ,,,,R-H,,,,,,,,,,,,,
ἦν,,,R--,,--H,,,,,,,,,,,,
οὗτος,,R--,,,,,,,,,,,,,,,
κακὸν,,,,,R--,R--,,,,,,,,,,,
ποιῶν,,,,,R--,R--,,,,,,,,,,,


In [39]:
pd.set_option('display.max_columns', None)

In [40]:
# this verse has no reference data! That seems wrong
sc.verse_dataframe("44005040")

,At,this,they,yielded,to,Gamaliel,They,called,the,apostles,in,and,had,them,flogged,Then,they.1,ordered,them.1,not,to.1,speak,in.1,the.1,name,of,Jesus,and.1,released,them.2
καὶ,,,--H,,,,,,,,,,,,,,,,,,,,,,,,,,,
προσκαλεσάμενοι,,,,--H,,,,,,,,,,,,,,,,,,,,,,,,,,
τοὺς,,,,,,,,,--H,,,,,,,,,,,,,,,,,,,,,
ἀποστόλους,,,,,,,,,,--H,,,,,,,,,,,,,,,,,,,,
δείραντες,,,,,,,,,,,,,,,--H,,,,,,,,,,,,,,,
παρήγγειλαν,,,,,,,,,,,,,,--H,,,,--H,--H,,,,,,,,,,,
μὴ,,,,,,,,,,,,,,,,,,,,--H,,,,,,,,,,
λαλεῖν,,,,,,,,,,,,,,,,,,,,,,--H,,,,,,,,
ἐπὶ,,,,,,,,,,,,,,,,,,,,,,,--H,,,,,,,
τῷ,,,,,,,,,,,,,,,,,,,,,,,,--H,,,,,,


In [47]:
sc.score_verse("44005040").hypothesispairs

[(<Source: 44005040001>, <Target: 44005040004>),
 (<Source: 44005040002>, <Target: 44005040005>),
 (<Source: 44005040003>, <Target: 44005040011>),
 (<Source: 44005040004>, <Target: 44005040012>),
 (<Source: 44005040005>, <Target: 44005040017>),
 (<Source: 44005040006>, <Target: 44005040016>),
 (<Source: 44005040006>, <Target: 44005040021>),
 (<Source: 44005040006>, <Target: 44005040022>),
 (<Source: 44005040007>, <Target: 44005040023>),
 (<Source: 44005040008>, <Target: 44005040025>),
 (<Source: 44005040009>, <Target: 44005040026>),
 (<Source: 44005040010>, <Target: 44005040027>),
 (<Source: 44005040011>, <Target: 44005040028>),
 (<Source: 44005040012>, <Target: 44005040029>),
 (<Source: 44005040013>, <Target: 44005040030>),
 (<Source: 44005040014>, <Target: 44005040032>),
 (<Source: 44005040015>, <Target: 44005040033>),
 (<Source: 44005040015>, <Target: 44005040034>)]